# NeuroScan Nepal — Standalone GPU Training

**Your local `src/` code is NOT modified.** This notebook uses a separate script.

1. **Runtime → Change runtime type → GPU (T4)**
2. Run all cells below

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Enable GPU first! Runtime → Change runtime type → GPU')

In [ ]:
!pip install -q opencv-python-headless scikit-learn matplotlib

In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile

WORK = Path('/content/neuroscan')
WORK.mkdir(exist_ok=True)
PRETRAINED = None

def find_data_on_drive():
    """Find folders that contain both normal/ and abnormal/ on Drive."""
    root = Path('/content/drive/MyDrive')
    candidates = []
    for abnormal in root.rglob('abnormal'):
        if not abnormal.is_dir():
            continue
        parent = abnormal.parent
        if (parent / 'normal').is_dir():
            candidates.append(parent)
    # prefer standard project layout: .../data/raw/
    for c in candidates:
        if c.name == 'raw' and c.parent.name == 'data':
            return c
    return candidates[0] if candidates else None

def find_zip_on_drive():
    return next((z for z in Path('/content/drive/MyDrive').rglob('neuroscan_data.zip') if z.is_file()), None)

drive.mount('/content/drive')

DATA_ROOT = find_data_on_drive()
if DATA_ROOT:
    print('Found dataset on Drive (no zip needed):', DATA_ROOT)
else:
    zip_path = find_zip_on_drive()
    if zip_path:
        print('Found zip on Drive:', zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(WORK)
    else:
        print('=' * 60)
        print('Upload neuroscan_data.zip from your PC:')
        print('  PC path: NeuroScan_Nepal\\scripts\\neuroscan_data.zip')
        print('  (Create it first: run zip_dataset_for_colab.bat)')
        print('When the button appears below, click Choose Files')
        print('=' * 60)
        uploaded = files.upload()
        while not uploaded:
            print('No file selected — click Choose Files and pick neuroscan_data.zip')
            uploaded = files.upload()
        name = next(iter(uploaded))
        zip_dest = WORK / 'neuroscan_data.zip'
        zip_dest.write_bytes(uploaded[name])
        print('Uploaded:', name, f'({len(uploaded[name]) / 1e6:.1f} MB)')
        with zipfile.ZipFile(zip_dest) as zf:
            zf.extractall(WORK)

    if (WORK / 'normal').exists():
        DATA_ROOT = WORK
    elif (WORK / 'raw' / 'normal').exists():
        DATA_ROOT = WORK / 'raw'
    elif (WORK / 'data' / 'raw' / 'normal').exists():
        DATA_ROOT = WORK / 'data' / 'raw'

if not DATA_ROOT or not (DATA_ROOT / 'normal').exists():
    raise FileNotFoundError(
        'Could not find normal/ and abnormal/ folders.\n'
        'On PC: run scripts/zip_dataset_for_colab.bat, then upload neuroscan_data.zip'
    )

for p in Path('/content/drive/MyDrive').rglob('cnn_baseline.pth'):
    PRETRAINED = p
    print('Found pretrained (optional):', PRETRAINED)
    break

print('Data root:', DATA_ROOT)
print('Normal images:', len(list((DATA_ROOT / 'normal').rglob('*'))))
print('Abnormal images:', len(list((DATA_ROOT / 'abnormal').rglob('*'))))

In [ ]:
# Upload neuroscan_colab_train.py from your PC:
# NeuroScan_Nepal/notebooks/colab/neuroscan_colab_train.py
uploaded = files.upload()
script = Path('neuroscan_colab_train.py')
for name, data in uploaded.items():
    script.write_bytes(data)
print('Script ready:', script.exists())

In [ ]:
pre = f'--pretrained {PRETRAINED}' if PRETRAINED and PRETRAINED.exists() else ''
fine = '--epochs 15 --lr 0.0001 --patience 5' if PRETRAINED and PRETRAINED.exists() else '--epochs 30 --lr 0.0005 --patience 7'
!python neuroscan_colab_train.py --data-root {DATA_ROOT} --out-dir /content/output --batch-size 64 {pre} {fine}

In [ ]:
from google.colab import files
import shutil
out = Path('/content/output')
drive_out = Path('/content/drive/MyDrive/NeuroScan/colab_output')
drive_out.mkdir(parents=True, exist_ok=True)
for f in out.iterdir():
    shutil.copy2(f, drive_out / f.name)
    files.download(str(f))
print('Also saved to Drive:', drive_out)